In [1]:
import pandas as pd
import numpy as np
import os
import glob as glob
import sys
import json

parent_dir = os.path.abspath(os.path.join(os.path.dirname(os.getcwd())))
sys.path.append(parent_dir)
import cpg_harmonizer
import s3_loader
import harmonize_checker

In [2]:
%load_ext autoreload
%autoreload 2

# Enter Identifiers, Per-Dataset Metadata

In [58]:
project_name = "cpg0030-gustafsdottir-cellpainting"
source_list = ['broad']
output_parent_directory = "/Users/eweisbar/Desktop/cpgtest"

# typically, per-dataset metadata
# if not dataset-wide, delete from here and create conditional entry below
# if unknown, comment out
per_dataset_manual = {
    'Plate_Size':384,
    'CP_Version':'v1',
    'DOI_to_Cite':'10.1371/journal.pone.0080999',
    'Year_Imaged':2010,
    'Cell_Line_Name':'U2OS',
    'Cell_Line_Type':'osteosarcoma',
    'Cell_Line_Modification':'None',
    'Microscope_Name':'Molecular Devices ImageXpress Micro',
    'Microscope_Binning': 2,
    'Microscope_Modality':'Widefield',
    'Microscope_Objective_Magnification':20,
    #'Microscope_Objective_NA':'', # not specified in publication
    #'Microscope_Pixel_Size': ,
    'Image_Bit_Depth': 12,
    'Image_Size_X':696,
    'Image_Size_Y':520,
    'Timepoint_Primary_Treatment':48,
    'Treatment_Category':'Compound',
    #'Timepoint_Secondary_Treatment':,
    #'Timepoint_Acquisition':,
}

In [ ]:
# excitation/emission values used for acquisition
# {label:{'ex_peak':value,'ex_width':value,'em_peak':value,'em_width':value}}
# if value is unknown, define value with np.nan
ex_em_dict = {
    'DNA':{'ex_peak':387,
           'ex_width':np.nan,
           'em_peak':447,
           'em_width':np.nan},
    'ER':{'ex_peak':472,
          'ex_width':np.nan,
           'em_peak':520,
           'em_width':np.nan},
    'RNA':{'ex_peak':531,
           'ex_width':np.nan,
           'em_peak':593,
           'em_width':np.nan},
    'AGP':{'ex_peak':562,
           'ex_width':np.nan,
           'em_peak':642,
           'em_width':np.nan},
    'Mito':{'ex_peak':628,
            'ex_width':np.nan,
           'em_peak':692,
           'em_width':np.nan}
}

# Join CPG Metadata

In [21]:
if len(source_list) == 1:
    output_directory = os.path.join(output_parent_directory, project_name, source_list[0], "workspace", "harmonized_metadata")
else:
    output_directory = os.path.join(output_parent_directory, project_name, "all", "workspace", "harmonized_metadata")
if not os.path.exists(output_directory):
    os.makedirs(output_directory, exist_ok=True)

metadata_paths = []
load_paths = []
for src in source_list:
    metadata_paths.extend(s3_loader.parse_s3_folder(f"{project_name}/{src}/workspace/metadata/platemaps/"))
    load_paths.extend(s3_loader.parse_s3_folder(f"{project_name}/{src}/workspace/load_data_csv/"))

project = cpg_harmonizer.Project(output_directory, "../output_structure.json", project_name=project_name)

In [22]:
main_csvs = []
main_csv_batch = []
for path in load_paths:
    if path.endswith("load_data.csv"):
        main_csvs.append(s3_loader.read_s3_file(path, sep = ","))
        main_csv_batch.append(path.split("/")[-3])

In [23]:
platemap_txt = []
platemap_csvs = []
platemap_csv_batch = []
platemap_txt_batch = []
platemap_txt_name = []
external_tsv = []


for path in metadata_paths:
    if path.endswith(".csv"):
        platemap_csvs.append(s3_loader.read_s3_file(path, sep = ","))
        platemap_csv_batch.append(path.split("/")[-2])
    if path.endswith(".txt"):
        platemap_txt.append(s3_loader.read_s3_file(path, sep = "\t"))
        platemap_txt_name.append(path.split("/")[-1].split(".")[0])
        platemap_txt_batch.append(path.split("/")[-3])
    if path.endswith(".tsv"):
        external_tsv.append(s3_loader.read_s3_file(path, sep = "\t"))

In [49]:
complete_df = project.run_conversion(
    main_csvs = main_csvs, 
    main_csv_batch = main_csv_batch, 
    platemap_csvs = platemap_csvs,  
    platemap_csv_batch = platemap_csv_batch, 
    platemap_txt= platemap_txt, 
    platemap_txt_batch = platemap_txt_batch,
    platemap_txt_name = platemap_txt_name,
    external_tsv = external_tsv
)

No external metadata found
Skipping external metadata
Harmonizing values of Treatment_Control_Class
Harmonizing values of Treatment_Solvent
Harmonizing values of Label


# Additional cleaning steps

In [50]:
with open('../inferable_relationships.json', "r") as f:
    inferable_metadata = json.load(f)

if per_dataset_manual['CP_Version'] != 'other':
    if not all([x in inferable_metadata["Label"].keys() for x in complete_df['Label'].unique()]):
        print(f'Labels need to be corrected to match any of {list(inferable_metadata["Label"].keys())}')
        print(f"Current labels are {complete_df['Label'].unique()}")
    # infer metadata from known relationships
    for column in inferable_metadata:
        for entry in inferable_metadata[column]:
            for inferred_column in inferable_metadata[column][entry]:
                if not inferred_column in complete_df.columns:
                    complete_df[inferred_column] = np.nan
                    complete_df[inferred_column] = complete_df[inferred_column].astype('str')
                complete_df.loc[complete_df[column]==entry, inferred_column] = inferable_metadata[column][entry][inferred_column]
else:
    print("Did not infer label metadata because CP_Version not inferrable")
    print(f"Labels that need to be manually declared are {complete_df['Label'].unique()}")

In [51]:
# add per label excitation and emission values
if set(complete_df['Label'].unique()) == ex_em_dict.keys():
    for key in ex_em_dict:
        complete_df.loc[complete_df["Label"] == key, "Microscope_Excitation_Peak"] = ex_em_dict[key]['ex_peak']
        complete_df.loc[complete_df["Label"] == key, "Microscope_Excitation_Width"] = ex_em_dict[key]['ex_width']
        complete_df.loc[complete_df["Label"] == key, "Microscope_Emission_Peak"] = ex_em_dict[key]['em_peak']
        complete_df.loc[complete_df["Label"] == key, "Microscope_Emission_Width"] = ex_em_dict[key]['em_width']

else:
    print("Excitation/Emission dictionary does not match Labels")
    print(f"Available labels are {set(complete_df['Label'].unique())}")
    print(f"Defined keys are {ex_em_dict.keys()}")

In [59]:
# add per-experiment metadata manually annotated above
for col, val in per_dataset_manual.items():
    complete_df[col] = val

# add source information inferred from file path
for source in source_list:
    complete_df.loc[complete_df['File Path'].str.contains(f"/{source}/"),'Source'] = source

In [61]:
# report on un-harmonized columns
# use reported information to manually update ontology OR dataframe OR input metadata
# this cell does NOT save the harmonization results, just check them
# this allows you to make any necessary corrections without accidentally overwriting
# returns a view of what the dataframe will look like after final harmonization
extra_cols = harmonize_checker.check_columns('../harmonized_ontology.json',complete_df, ret="extra_cols")

print("View of data that will be kept")
complete_df[[x for x in complete_df.columns if x not in extra_cols]]

View of data that will be kept


,Treatment_SMILES,Well,Plate,Treatment_Primary_Treatment,Site,Batch,File Path,File Name,Treatment_Broad_Sample,Treatment_Solvent,...,Treatment_PubChem_CID,Treatment_Mechanism,Timepoint_Acquisition,Microscope_Objective_NA,Timepoint_Secondary_Treatment,Image_Position_Z,Treatment_InChIKey,Year_Imaged,Treatment_Secondary_Treatment,Microscope_Pixel_Size
0,OC(=O)c1cccnc1Nc2cccc(c2)C(F)(F)F,A01,20585,NIFLUMIC ACID,1,BBBC022,https://cellpainting-gallery.s3.us-east-1.amaz...,IXMtest_A01_s1_w3FD54227A-6BE4-460B-8E2C-54B87...,BRD-K98763141-001-06-8,DMSO,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2010,<NA>,<NA>
1,OC(=O)c1cccnc1Nc2cccc(c2)C(F)(F)F,A01,20585,NIFLUMIC ACID,2,BBBC022,https://cellpainting-gallery.s3.us-east-1.amaz...,IXMtest_A01_s2_w3A597237B-C3D7-43AE-8399-83E76...,BRD-K98763141-001-06-8,DMSO,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2010,<NA>,<NA>
2,OC(=O)c1cccnc1Nc2cccc(c2)C(F)(F)F,A01,20585,NIFLUMIC ACID,3,BBBC022,https://cellpainting-gallery.s3.us-east-1.amaz...,IXMtest_A01_s3_w30B946780-C923-4A40-8EC5-16262...,BRD-K98763141-001-06-8,DMSO,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2010,<NA>,<NA>
3,OC(=O)c1cccnc1Nc2cccc(c2)C(F)(F)F,A01,20585,NIFLUMIC ACID,4,BBBC022,https://cellpainting-gallery.s3.us-east-1.amaz...,IXMtest_A01_s4_w3F582BADD-2249-48D6-BC19-1B4C1...,BRD-K98763141-001-06-8,DMSO,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2010,<NA>,<NA>
4,OC(=O)c1cccnc1Nc2cccc(c2)C(F)(F)F,A01,20585,NIFLUMIC ACID,5,BBBC022,https://cellpainting-gallery.s3.us-east-1.amaz...,IXMtest_A01_s5_w3EB7ABD45-EE05-41A0-979E-7F305...,BRD-K98763141-001-06-8,DMSO,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2010,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
345595,Oc1ccc(cc1)[C@@H]2CC(=O)c3c(O)cc(O)cc3O2,P24,20646,NARINGENIN,5,BBBC022,https://cellpainting-gallery.s3.us-east-1.amaz...,IXMtest_P24_s5_w23EE5080A-0B6E-4AE1-A413-A51C4...,BRD-K08832567-001-02-4,DMSO,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2010,<NA>,<NA>
345596,Oc1ccc(cc1)[C@@H]2CC(=O)c3c(O)cc(O)cc3O2,P24,20646,NARINGENIN,6,BBBC022,https://cellpainting-gallery.s3.us-east-1.amaz...,IXMtest_P24_s6_w2F9235A94-86DA-4703-B46B-DAC2C...,BRD-K08832567-001-02-4,DMSO,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2010,<NA>,<NA>
345597,Oc1ccc(cc1)[C@@H]2CC(=O)c3c(O)cc(O)cc3O2,P24,20646,NARINGENIN,7,BBBC022,https://cellpainting-gallery.s3.us-east-1.amaz...,IXMtest_P24_s7_w209CDD7FD-92F0-40B4-85C6-E3795...,BRD-K08832567-001-02-4,DMSO,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2010,<NA>,<NA>
345598,Oc1ccc(cc1)[C@@H]2CC(=O)c3c(O)cc(O)cc3O2,P24,20646,NARINGENIN,8,BBBC022,https://cellpainting-gallery.s3.us-east-1.amaz...,IXMtest_P24_s8_w23695868E-F144-4CDF-B1A6-64C36...,BRD-K08832567-001-02-4,DMSO,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2010,<NA>,<NA>


In [62]:
print("View of data that will be removed")
complete_df[extra_cols]

View of data that will be removed


""
0
1
2
3
4
...
345595
345596
345597
345598


In [63]:
# After correcting any warnings above, run to save harmonization
complete_df = harmonize_checker.check_columns('../harmonized_ontology.json',complete_df)

In [64]:
saved_path = os.path.join(output_directory,f"{project_name}_harmonized_metadata.parquet")
complete_df.to_parquet(saved_path)